# DOCX to Markdown Pipeline

Converts a structured functional-requirements DOCX to clean markdown.

## How headings are detected

In Word the document renders as:
- `1. Login`, `2. Site Navigation` … → `#` (H1)
- `A. Purpose`, `B. Core Functional Requirements` … → `##` (H2)
- Sub-items → `###` (H3)

## Key functions and what they solve

| Function | Purpose |
|---|---|
| `is_sentence_like` | Guards against promoting full sentences to headings |
| `is_inline_heading_candidate` | Detects `Label: detail sentence` — promotes to heading + body |
| `is_long_lead_in` | Detects long intro phrases like `Button to download… for:` — becomes `list_intro`, not heading |
| `is_inventory_sequence_item` | Prevents flat bullet-list peers becoming headings |
| `is_terminal_short_list_item` | Prevents last item in a list becoming a heading when a heading follows |
| `should_be_group_label` | Turns deep (ilvl≥3) short labels into `list_intro` rather than headings |
| `is_heading_candidate` | Main gate: checks num_level, inline, short, next-block context |
| `classify_heading_level` | Decides H2 vs H3 based on h2_signature |
| `should_be_list_intro` | Downgrades certain H3 candidates to `list_intro` |
| `split_heading_block` | Splits `Label: detail` into heading + body paragraph |

## 1) Imports and constants

In [11]:
from __future__ import annotations

import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterator, Literal, Optional

from docx import Document
from docx.document import Document as _Document
from docx.oxml.table import CT_Tbl
from docx.oxml.text.paragraph import CT_P
from docx.oxml.text.run import CT_R
from docx.table import Table, _Cell
from docx.text.paragraph import Paragraph
from lxml import etree

# Only for 'A./B./C./D.' items in Data upload page (Normal bold, no num_id, letter in text)
RE_ALPHA_SECTION = re.compile(r"^\s*(?P<marker>[A-Z])\s*\.\s*\.?\s*(?P<body>.+?)\s*$")
RE_NUMERIC_SECTION = re.compile(r"^\s*(?P<marker>\d+)\s*\.\s*(?P<body>.+?)\s*$")

# 'Short label: detail sentence'
RE_INLINE_LABEL = re.compile(r"^(?P<label>[^:]{1,80}?)\s*:\s*(?P<detail>.+)$")

# Plain-text list markers: '1) Item', 'a) Item'
RE_TEXT_LIST = re.compile(r"^\s*(?P<marker>(?:\d+|[A-Za-z])[.)])\s+(?P<body>.+?)\s*$")

NS_R = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"

## 2) Data models

In [12]:
@dataclass
class RawBlock:
    """One paragraph, table or image extracted directly from the DOCX."""
    kind: Literal["paragraph", "table", "image"]
    text: str = ""
    style_name: Optional[str] = None
    num_id: Optional[int] = None
    num_level: Optional[int] = None
    is_boldish: bool = False
    position: int = 0
    table_rows: list[list[str]] = field(default_factory=list)
    image_filename: Optional[str] = None


@dataclass
class TypedBlock:
    """A RawBlock classified into a semantic role."""
    block_type: Literal[
        "heading", "paragraph", "list_intro",
        "ordered_list_item", "unordered_list_item",
        "table", "image", "ignore",
    ]
    text: str = ""
    heading_level: Optional[int] = None
    list_level: Optional[int] = None
    ordered: bool = False
    raw: Optional[RawBlock] = None
    reasons: list[str] = field(default_factory=list)


@dataclass
class Node:
    """A node in the document tree."""
    node_type: Literal["root", "section", "paragraph", "list_intro",
                       "list", "list_item", "table", "image"]
    text: str = ""
    level: int = 0
    ordered: bool = False
    children: list["Node"] = field(default_factory=list)
    table_rows: list[list[str]] = field(default_factory=list)
    image_filename: Optional[str] = None

    def add_child(self, child: "Node") -> None:
        self.children.append(child)

## 3) DOCX low-level helpers

In [ ]:
def iter_block_items(parent: _Document | _Cell) -> Iterator[Paragraph | Table]:
    """Yield paragraphs and tables in document order."""
    parent_elm = parent.element.body if isinstance(parent, _Document) else parent._tc
    
    for child in parent_elm.iterchildren():
        if isinstance(child, CT_P):
            yield Paragraph(child, parent)
        elif isinstance(child, CT_Tbl):
            yield Table(child, parent)


def clean_text(text: str) -> str:
    text = text.replace("\xa0", " ").replace("\u200b", "")
    return re.sub(r"\s+", " ", text).strip()


def get_style_name(p: Paragraph) -> Optional[str]:
    try:
        return p.style.name if p.style else None
    except Exception:
        return None


def is_boldish(p: Paragraph) -> bool:
    runs = [r for r in p.runs if clean_text(r.text)]
    if not runs:
        return False
    return sum(1 for r in runs if bool(r.bold)) >= max(1, (len(runs) + 1) // 2)


def get_num_info(p: Paragraph) -> tuple[Optional[int], Optional[int]]:
    ppr = p._p.pPr
    if ppr is None or ppr.numPr is None:
        return None, None
    np = ppr.numPr
    level  = int(np.ilvl.val)  if np.ilvl  is not None else 0
    num_id = int(np.numId.val) if np.numId is not None else None
    return num_id, level


def resolve_list_kind(document: _Document, num_id: Optional[int]) -> Optional[str]:
    """Return 'ordered' or 'unordered' by inspecting abstractNum for num_id."""
    if num_id is None:
        return None
    numbering_part = getattr(document.part, "numbering_part", None)
    if numbering_part is None:
        return None
    root = etree.fromstring(numbering_part.blob)
    ns   = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    nums = root.xpath(f'.//w:num[@w:numId="{num_id}"]', namespaces=ns)
    if not nums:
        return None
    abstract_ids = nums[0].xpath("./w:abstractNumId/@w:val", namespaces=ns)
    if not abstract_ids:
        return None
    abstract_nodes = root.xpath(
        f'.//w:abstractNum[@w:abstractNumId="{abstract_ids[0]}"]', namespaces=ns
    )
    if not abstract_nodes:
        return None
    formats = abstract_nodes[0].xpath(".//w:lvl[1]/w:numFmt/@w:val", namespaces=ns)
    if not formats:
        return None
    return "unordered" if formats[0].lower() == "bullet" else "ordered"


def get_heading_level_from_style(style_name: Optional[str]) -> Optional[int]:
    if not style_name:
        return None
    m = re.match(r"Heading\s+(\d+)$", style_name, flags=re.I)
    return int(m.group(1)) if m else None


def get_image_extension(image_part) -> str:
    try:
        suffix = Path(str(image_part.partname)).suffix.lower()
        if suffix:
            return suffix
    except Exception:
        pass
    return ".png"


def iter_paragraph_segments(
    p: Paragraph, image_dir: Path, doc_stem: str, counter: list[int]
) -> list[dict]:
    """
    Walk paragraph runs in order, returning [{kind:'text'|'image', ...}].
    Images are saved to image_dir immediately.
    """
    segments: list[dict] = []
    buf: list[str] = []

    def flush():
        t = clean_text("".join(buf))
        if t:
            segments.append({"kind": "text", "text": t})
        buf.clear()

    for child in p._p.iterchildren():
        if not isinstance(child, CT_R):
            continue
        for rc in child.iterchildren():
            tag = etree.QName(rc.tag).localname
            if tag == "t":
                buf.append(rc.text or "")
            elif tag in {"tab", "br", "cr"}:
                buf.append(" ")
            elif tag == "drawing":
                for elem in rc.iter():
                    if etree.QName(elem.tag).localname != "blip":
                        continue
                    rel_id = elem.get(f"{{{NS_R}}}embed")
                    if not rel_id:
                        continue
                    image_part = p.part.related_parts[rel_id]
                    ext = get_image_extension(image_part)
                    counter[0] += 1
                    filename = f"{doc_stem}_image_{counter[0]:03d}{ext}"
                    image_dir.mkdir(parents=True, exist_ok=True)
                    (image_dir / filename).write_bytes(image_part.blob)
                    flush()
                    segments.append({"kind": "image", "filename": filename})
    flush()
    return segments

## 4) Raw block extraction

In [14]:
def extract_raw_blocks(docx_path: Path, image_dir: Path) -> list[RawBlock]:
    """Extract every paragraph, table and image from the DOCX in document order."""
    document = Document(str(docx_path))
    doc_stem = docx_path.stem.lower().replace(" ", "_")
    blocks: list[RawBlock] = []
    position = 0
    counter  = [0]

    print(document)
    for item in iter_block_items(document):

        if isinstance(item, Table):
            rows = [[clean_text(cell.text) for cell in row.cells] for row in item.rows]
            rows = [r for r in rows if any(r)]
            if rows:
                blocks.append(RawBlock(kind="table", table_rows=rows, position=position))
            position += 1
            continue

        segments = iter_paragraph_segments(item, image_dir, doc_stem, counter)

        if not segments:
            text = clean_text(item.text)
            if not text:
                position += 1
                continue
            num_id, num_level = get_num_info(item)
            blocks.append(RawBlock(
                kind="paragraph",
                text=text,
                style_name=get_style_name(item),
                num_id=num_id,
                num_level=num_level,
                is_boldish=is_boldish(item),
                position=position,
            ))
            position += 1
            continue

        num_id, num_level = get_num_info(item)
        bold = is_boldish(item)
        for local_pos, seg in enumerate(segments):
            if seg["kind"] == "text":
                blocks.append(RawBlock(
                    kind="paragraph",
                    text=seg["text"],
                    style_name=get_style_name(item),
                    num_id=num_id,
                    num_level=num_level,
                    is_boldish=bold,
                    position=position + local_pos,
                ))
            else:
                blocks.append(RawBlock(
                    kind="image",
                    image_filename=seg["filename"],
                    style_name=get_style_name(item),
                    position=position + local_pos,
                ))
        position += max(len(segments), 1)

    return blocks


def normalize_raw_blocks(blocks: list[RawBlock]) -> list[RawBlock]:
    """Merge continuation lines: same style, no num, starts lowercase,
    previous line does not end with sentence-ending punctuation."""
    out: list[RawBlock] = []
    for block in blocks:
        if block.kind in {"table", "image"} or not out or out[-1].kind != "paragraph":
            out.append(block)
            continue
        prev = out[-1]
        can_merge = (
            prev.num_id is None
            and block.num_id is None
            and get_heading_level_from_style(prev.style_name) is None
            and get_heading_level_from_style(block.style_name) is None
            and prev.style_name == block.style_name
            and not prev.text.endswith((".", "!", "?", ":"))
            and block.text[:1].islower()
        )
        if can_merge:
            prev.text = clean_text(prev.text + " " + block.text)
        else:
            out.append(block)
    return out


def drop_preface(blocks: list[RawBlock], start_heading_text: str = "Login") -> list[RawBlock]:
    """Drop everything before the first Heading 2 matching start_heading_text."""
    target = start_heading_text.lower()
    for idx, block in enumerate(blocks):
        if block.kind == "paragraph" and get_heading_level_from_style(block.style_name) == 2:
            if not target or clean_text(block.text).lower() == target:
                return blocks[idx:]
    for idx, block in enumerate(blocks):
        if block.kind == "paragraph" and get_heading_level_from_style(block.style_name) == 2:
            return blocks[idx:]
    return blocks

## 5) Heading-detection helpers

These are the exact functions the original pipeline used, preserved faithfully.
Each one solves a specific edge case — removing any of them breaks real items.

In [15]:
def parse_section_marker(text: str) -> tuple[Optional[str], Optional[str]]:
    """Detect 'A. Body' or '1. Body' in the text string itself.
    Only fires for the 4 items in Data upload page that literally have the letter in the text.
    All other A./B. headings use DOCX numbering and never have the prefix in the text.
    """
    m = RE_NUMERIC_SECTION.match(text)
    if m:
        return "numeric", clean_text(m.group("body"))
    m = RE_ALPHA_SECTION.match(text)
    if m:
        return "alpha", clean_text(m.group("body"))
    return None, None


def split_inline_label(text: str) -> tuple[Optional[str], Optional[str]]:
    """Split 'Short label: detail sentence' → (label, detail) or (None, None)."""
    m = RE_INLINE_LABEL.match(clean_text(text))
    if not m:
        return None, None
    label  = clean_text(m.group("label"))
    detail = clean_text(m.group("detail"))
    if not label or not detail:
        return None, None
    if len(label) < 3 or len(label.split()) > 8:
        return None, None
    return label, detail


def is_inline_heading_candidate(text: str) -> bool:
    """True when text is 'Short label: at least 3-word detail'.
    e.g. 'Assessment selection : Users can select one or more…'
    These become heading + body paragraph.
    """
    label, detail = split_inline_label(text)
    if label is None or detail is None:
        return False
    return len(detail.split()) >= 3


def short_label_text(text: str) -> str:
    """The label part of text: strips section markers, inline detail, trailing colon."""
    marker_kind, marker_body = parse_section_marker(text)
    if marker_kind is not None and marker_body is not None:
        return marker_body
    label, _ = split_inline_label(text)
    if label is not None:
        return label
    return clean_text(text).rstrip(":").strip()


def label_word_count(text: str) -> int:
    """Word count of the label part, ignoring parenthetical notes.
    e.g. 'Overall cost (needs GL)' → 2 words (not 4).
    This keeps 'Overall cost (needs GL):' correctly classified as a short label.
    """
    label = re.sub(r"\([^)]*\)", "", short_label_text(text))
    return len(clean_text(label).split())


def is_sentence_like(text: str) -> bool:
    """True when text reads as a sentence rather than a title."""
    words = clean_text(text).split()
    if len(words) >= 10:
        return True
    if text.endswith((".", "?", "!")):
        return True
    if re.search(
        r"\b(is|are|should|will|can|must|provides|allows|captures|means|shows)\b",
        text, re.I
    ):
        return True
    return False


def is_short_label(text: str) -> bool:
    """True when text looks like a section title: short, not sentence-like.
    Uses short_label_text (strips parens and detail) so 'Overall cost (needs GL):'
    is 2 label-words and correctly returns True.
    """
    label = short_label_text(text)
    if not label or len(label) > 90:
        return False
    if label_word_count(text) > 5:
        return False
    if is_sentence_like(label):
        return False
    return True


def block_signature(block: RawBlock) -> tuple:
    """Fingerprint used to decide if a List Paragraph is H2 or H3.
    Two blocks with the same signature → same heading level.
    """
    marker_kind, _ = parse_section_marker(block.text)
    if marker_kind:
        return ("section-marker", marker_kind)
    if block.num_id is not None:
        return ("numbering", (block.num_id, block.num_level or 0))
    if block.style_name:
        return ("style", block.style_name.lower())
    return ("plain", "plain")


def is_long_lead_in(block: RawBlock, next_block: Optional[RawBlock]) -> bool:
    """True when a block is a long intro phrase that leads into a list.
    e.g. 'Button to download sample/template files for:' (6 label-words) followed
    by a numbered list → becomes list_intro, not a heading.
    Guards against promoting verbose labels to headings.
    """
    if next_block is None:
        return False
    if parse_section_marker(block.text)[0] is not None:
        return False
    if not clean_text(block.text).endswith(":"):
        return False
    if label_word_count(block.text) <= 4:   # short labels remain heading candidates
        return False
    if (
        block.num_id is not None
        and next_block.num_id == block.num_id
        and (next_block.num_level or 0) > (block.num_level or 0)
    ):
        return False
    return (
        next_block.num_id is not None
        or RE_TEXT_LIST.match(next_block.text) is not None
        or is_short_label(next_block.text)
    )


def looks_like_heading_boundary(block: Optional[RawBlock]) -> bool:
    if block is None:
        return False
    if get_heading_level_from_style(block.style_name) in {2, 3}:
        return True
    if parse_section_marker(block.text)[0] is not None:
        return True
    return False


def is_terminal_short_list_item(
    block: RawBlock,
    prev_typed: Optional[TypedBlock],
    next_block: Optional[RawBlock],
) -> bool:
    """True when a short list item is the last item before a heading boundary.
    Prevents it from being promoted to a heading.
    """
    if block.num_id is None or next_block is None or prev_typed is None:
        return False
    if not is_short_label(block.text):
        return False
    if clean_text(block.text).endswith(":"):
        return False
    if prev_typed.block_type not in {"ordered_list_item", "unordered_list_item"}:
        return False
    return looks_like_heading_boundary(next_block)


def is_inventory_sequence_item(
    block: RawBlock,
    prev_block: Optional[RawBlock],
    next_block: Optional[RawBlock],
) -> bool:
    """True when a numbered short-label belongs to a flat enumeration list
    (same num_id+level peers on both sides, no deeper child body below it).
    Prevents flat bullet peers from becoming headings.
    e.g. 'Benchmark Upload', 'Explore Benchmarks', 'Output' under ## Screens.
    """
    if block.num_id is None or not is_short_label(block.text):
        return False
    if is_inline_heading_candidate(block.text):
        return False

    same_prev = (
        prev_block is not None
        and prev_block.num_id    == block.num_id
        and prev_block.num_level == block.num_level
        and is_short_label(prev_block.text)
        and not is_sentence_like(prev_block.text)
    )
    same_next = (
        next_block is not None
        and next_block.num_id    == block.num_id
        and next_block.num_level == block.num_level
        and is_short_label(next_block.text)
        and not is_sentence_like(next_block.text)
    )
    followed_by_body = (
        next_block is not None
        and (
            (next_block.num_id == block.num_id and (next_block.num_level or 0) > (block.num_level or 0))
            or (next_block.num_id is None and is_sentence_like(next_block.text))
        )
    )
    return (same_prev or same_next) and not followed_by_body


def should_be_group_label(block: RawBlock, next_block: Optional[RawBlock]) -> bool:
    """True for deep (ilvl >= 3) short labels that head a sub-group.
    These become list_intro rather than ### headings.
    e.g. 'Headcount/Outsource' at ilvl=4 → list_intro.
    """
    if block.num_id is None or next_block is None:
        return False
    if (block.num_level or 0) < 3:
        return False
    if not is_short_label(block.text):
        return False
    next_is_peer_or_child_body = (
        next_block.num_id is not None
        and (
            (next_block.num_level or 0) >= (block.num_level or 0)
            or (next_block.num_level or 0) == (block.num_level or 0) - 1
        )
        and is_sentence_like(next_block.text)
    )
    next_is_child = (
        next_block.num_id is not None
        and (next_block.num_level or 0) > (block.num_level or 0)
    )
    return next_is_peer_or_child_body or next_is_child


def is_heading_candidate(
    block: RawBlock,
    prev_block: Optional[RawBlock],
    next_block: Optional[RawBlock],
    current_heading_level: int,
    current_h2_signature: Optional[tuple],
) -> bool:
    """Main gate: decides whether a block is a heading candidate.

    For numbered blocks (num_id is not None):
      - Immediately rejects ilvl > 2 (those are deep list items, never headings)
      - At ilvl 0: requires short_label OR inline_heading_candidate, plus next-block context
      - At ilvl 1-2: same conditions but stricter

    For un-numbered blocks (no num_id):
      - Must be short_label AND end with ':' AND have a next block
      - (Figma TBD) fails because it does not end with ':'
      - 'Supported Formats:xlsx' fails because short_label_text strips the inline
        detail, leaving just 'Supported Formats' — but the text does NOT end with ':'
        (it ends with 'xlsx'), so it falls through to paragraph
    """
    if block.kind != "paragraph":
        return False
    if block.style_name == "Caption":
        return False

    marker_kind, marker_body = parse_section_marker(block.text)
    if marker_kind is not None and marker_body is not None:
        return True

    if is_long_lead_in(block, next_block):
        return False

    if is_inventory_sequence_item(block, prev_block, next_block):
        return False

    inline = is_inline_heading_candidate(block.text)
    short  = is_short_label(block.text)

    if block.num_id is not None:
        # ilvl > 2: these are always deep list items, never headings
        if (block.num_level or 0) > 2:
            return False
        
        if inline and short:
            return True

        if next_block is None:
            return False
        if get_heading_level_from_style(next_block.style_name) == 2:
            return False

        next_is_short_label        = is_short_label(next_block.text)
        next_is_same_level_label   = (next_block.num_id == block.num_id and next_block.num_level == block.num_level and next_is_short_label)
        next_is_same_level_body    = (next_block.num_id == block.num_id and next_block.num_level == block.num_level and is_sentence_like(next_block.text))
        next_is_other_num_label    = (next_block.num_id is not None and next_block.num_id != block.num_id and next_is_short_label)
        next_is_other_num_body     = (next_block.num_id is not None and next_block.num_id != block.num_id and is_sentence_like(next_block.text))
        next_is_other_num_intro    = (next_block.num_id is not None and next_block.num_id != block.num_id and clean_text(next_block.text).endswith(":"))
        next_is_deeper             = (next_block.num_id is not None and (next_block.num_level or 0) > (block.num_level or 0))
        next_is_adjacent_body_list = (next_block.num_id is not None and (next_block.num_level or 0) == (block.num_level or 0) and is_sentence_like(next_block.text))
        next_is_body               = (next_block.num_id is None and get_heading_level_from_style(next_block.style_name) is None)

        if (block.num_level or 0) == 0:
            if clean_text(block.text).endswith(":") and not short:
                return next_is_same_level_label or next_is_other_num_label or next_is_deeper
            return short and (
                next_is_same_level_label or next_is_same_level_body
                or next_is_other_num_label or next_is_other_num_body
                or next_is_other_num_intro or next_is_adjacent_body_list
                or next_is_deeper or next_is_body
            )

        # ilvl 1 or 2
        return short and (
            next_is_same_level_label or next_is_same_level_body
            or next_is_other_num_label or next_is_other_num_body
            or next_is_other_num_intro or next_is_adjacent_body_list
            or next_is_deeper or next_is_body
        )

    if not short:
        return False
    if clean_text(block.text).endswith(":"):
        return next_block is not None
    return False


def should_be_list_intro(block: RawBlock, current_heading_level: int) -> bool:
    """Downgrade a heading candidate to list_intro when:
    - we are already at depth 3 (current_heading_level >= 3)
    - the block has a num_id at ilvl > 0
    - it is a short label but NOT an inline heading candidate
    This prevents over-deep heading nesting.
    """
    if current_heading_level < 3:
        return False
    if block.num_id is None:
        return False
    if (block.num_level or 0) == 0:
        return False
    if clean_text(block.text).endswith(":") or is_inline_heading_candidate(block.text):
        return False
    return is_short_label(block.text)


def classify_heading_level(
    block: RawBlock,
    current_h2_signature: Optional[tuple],
    current_heading_level: int,
) -> int:
    """Return 1, 2 or 3 for the heading level.
    Section markers: numeric → 1, alpha → 2.
    Otherwise: first heading after H1 reset → 2 (sets h2_signature).
    Same signature as h2 → 2. Different → 3.
    """
    marker_kind, _ = parse_section_marker(block.text)
    if marker_kind == "numeric":
        return 1
    if marker_kind == "alpha":
        return 2
    sig = block_signature(block)
    if current_h2_signature is None:
        return 2
    if sig == current_h2_signature:
        return 2
    if current_heading_level <= 1:
        return 2
    return 3


def split_heading_block(block: RawBlock, heading_level: int) -> list[TypedBlock]:
    """Produce heading TypedBlock(s) from a block.
    For 'Label: detail sentence' blocks: heading + body paragraph.
    """
    label, detail = split_inline_label(block.text)
    marker_kind, marker_body = parse_section_marker(block.text)
    heading_text = marker_body or label or clean_text(block.text).rstrip(":").strip()

    results = [TypedBlock(
        block_type="heading",
        text=heading_text,
        heading_level=heading_level,
        raw=block,
        reasons=["structural heading"],
    )]
    if detail:
        results.append(TypedBlock(
            block_type="paragraph",
            text=detail,
            raw=block,
            reasons=["split inline heading detail"],
        ))
    return results

## 6) Block classification

Processing order matters — checks are evaluated top to bottom and `continue` on first match:

1. Table / image / caption → pass-through
2. Explicit DOCX Heading style (H2→`#`, H3→`##`)
3. `is_long_lead_in` → `list_intro` (catches verbose phrases like 'Button to download… for:')
4. `is_terminal_short_list_item` → `unordered_list_item`
5. `is_heading_candidate` → heading or `list_intro` (via `should_be_list_intro`)
6. `should_be_group_label` → `list_intro` (deep ilvl≥3 group labels like 'Headcount/Outsource')
7. `is_inventory_sequence_item` → `unordered_list_item`
8. Numbered body paragraph special case
9. DOCX list kind → `ordered_list_item` / `unordered_list_item`
10. Plain-text list markers
11. Default: paragraph

In [16]:
def classify_blocks(
    docx_path: Path,
    blocks: list[RawBlock],
    max_heading_level: int = 3,
) -> list[TypedBlock]:

    document = Document(str(docx_path))
    typed: list[TypedBlock] = []

    current_heading_level: int = 0
    current_h2_signature: Optional[tuple] = None

    for index, block in enumerate(blocks):
        prev_block = blocks[index - 1] if index > 0 else None
        next_block = blocks[index + 1] if index + 1 < len(blocks) else None
        prev_typed = typed[-1] if typed else None

        if block.kind == "table":
            typed.append(TypedBlock(block_type="table", raw=block))
            continue

        if block.kind == "image":
            typed.append(TypedBlock(block_type="image", raw=block))
            continue

        if block.style_name == "Caption":
            typed.append(TypedBlock(block_type="ignore", raw=block, reasons=["caption"]))
            continue

        style_level = get_heading_level_from_style(block.style_name)

        # Heading 2 = top-level sections (renders as '1. Login', '2. Site Navigation'…)
        if style_level == 2:
            current_heading_level = 1
            current_h2_signature  = None
            typed.append(TypedBlock(
                block_type="heading",
                text=clean_text(block.text),
                heading_level=1,
                raw=block,
                reasons=["Heading 2 -> H1"],
            ))
            continue

        if style_level == 3:
            current_heading_level = 2
            current_h2_signature  = ("style-heading-3", "style-heading-3")
            typed.append(TypedBlock(
                block_type="heading",
                text=clean_text(block.text).rstrip(":"),
                heading_level=2,
                raw=block,
                reasons=["Heading 3 -> H2"],
            ))
            continue

        if is_long_lead_in(block, next_block):
            label, detail = split_inline_label(block.text)
            intro_text = label or short_label_text(block.text)
            typed.append(TypedBlock(
                block_type="list_intro",
                text=intro_text.rstrip(":").strip(),
                raw=block,
                reasons=["long lead-in"],
            ))
            if detail and detail != intro_text:
                typed.append(TypedBlock(
                    block_type="paragraph",
                    text=detail,
                    raw=block,
                    reasons=["split lead-in detail"],
                ))
            continue

        if is_terminal_short_list_item(block, prev_typed, next_block):
            typed.append(TypedBlock(
                block_type="unordered_list_item",
                text=short_label_text(block.text),
                list_level=0,
                ordered=False,
                raw=block,
                reasons=["terminal short list item"],
            ))
            continue

        if is_heading_candidate(block, prev_block, next_block,
                                current_heading_level, current_h2_signature):

            heading_level = min(
                classify_heading_level(block, current_h2_signature, current_heading_level),
                max_heading_level,
            )

            if heading_level > max_heading_level or (
                heading_level == max_heading_level
                and should_be_list_intro(block, current_heading_level)
            ):
                label, detail = split_inline_label(block.text)
                intro_text = label or short_label_text(block.text)
                typed.append(TypedBlock(
                    block_type="list_intro",
                    text=intro_text.rstrip(":").strip(),
                    raw=block,
                    reasons=["deep label downgraded to list intro"],
                ))
                if detail:
                    typed.append(TypedBlock(
                        block_type="paragraph",
                        text=detail,
                        raw=block,
                        reasons=["split inline list-intro detail"],
                    ))
                continue

            heading_blocks = split_heading_block(block, heading_level)
            typed.extend(heading_blocks)
            current_heading_level = heading_level
            if heading_level == 2:
                current_h2_signature = block_signature(block)
            continue

        if should_be_group_label(block, next_block):
            typed.append(TypedBlock(
                block_type="list_intro",
                text=short_label_text(block.text),
                raw=block,
                reasons=["deep group label"],
            ))
            continue

        if is_inventory_sequence_item(block, prev_block, next_block):
            typed.append(TypedBlock(
                block_type="unordered_list_item",
                text=short_label_text(block.text),
                list_level=0,
                ordered=False,
                raw=block,
                reasons=["inventory sequence"],
            ))
            continue

        list_kind = resolve_list_kind(document, block.num_id)
        next_looks_like_heading = (
            next_block is not None
            and (
                looks_like_heading_boundary(next_block)
                or (
                    next_block.num_id is not None
                    and is_short_label(next_block.text)
                    and not is_inventory_sequence_item(
                        next_block, block,
                        blocks[index + 2] if index + 2 < len(blocks) else None
                    )
                )
            )
        )
        if (
            block.num_id is not None
            and list_kind != "ordered"
            and is_sentence_like(block.text)
            and not clean_text(block.text).endswith(":")
            and typed
            and prev_typed is not None
            and prev_typed.block_type in {"heading", "paragraph"}
            and (
                prev_typed.block_type == "heading"
                or "split inline heading detail" in prev_typed.reasons
            )
            and next_looks_like_heading
        ):
            typed.append(TypedBlock(
                block_type="paragraph",
                text=clean_text(block.text),
                raw=block,
                reasons=["numbered body paragraph"],
            ))
            continue

        if list_kind is not None:
            typed.append(TypedBlock(
                block_type="ordered_list_item" if list_kind == "ordered" else "unordered_list_item",
                text=clean_text(block.text),
                list_level=max(block.num_level or 0, 0),
                ordered=(list_kind == "ordered"),
                raw=block,
                reasons=["docx numbering"],
            ))
            continue

        m = RE_TEXT_LIST.match(block.text)
        if m:
            marker = m.group("marker")
            typed.append(TypedBlock(
                block_type="ordered_list_item" if marker[0].isdigit() else "unordered_list_item",
                text=clean_text(m.group("body")),
                list_level=0,
                ordered=marker[0].isdigit(),
                raw=block,
                reasons=["text list marker"],
            ))
            continue

        typed.append(TypedBlock(
            block_type="paragraph",
            text=clean_text(block.text),
            raw=block,
            reasons=["default paragraph"],
        ))

    return typed

## 7) Tree construction

In [17]:
def build_tree(typed_blocks: list[TypedBlock]) -> Node:
    root = Node(node_type="root")
    section_stack: list[Node] = [root]
    list_stack: list[tuple[int, bool, Node]] = []
    current_intro: Optional[Node] = None

    def current_parent() -> Node:
        return current_intro if current_intro is not None else section_stack[-1]

    def clear_list_context() -> None:
        nonlocal current_intro
        list_stack.clear()
        current_intro = None

    def last_list_item() -> Optional[Node]:
        if not list_stack:
            return None
        last = list_stack[-1][2].children[-1] if list_stack[-1][2].children else None
        return last if last and last.node_type == "list_item" else None

    for tb in typed_blocks:
        if tb.block_type == "ignore":
            continue

        if tb.block_type == "table":
            clear_list_context()
            section_stack[-1].add_child(Node(node_type="table", table_rows=tb.raw.table_rows if tb.raw else []))
            continue

        if tb.block_type == "image":
            clear_list_context()
            section_stack[-1].add_child(Node(node_type="image", image_filename=tb.raw.image_filename if tb.raw else None))
            continue

        if tb.block_type == "heading":
            clear_list_context()
            level = tb.heading_level or 1
            while len(section_stack) > 1 and section_stack[-1].level >= level:
                section_stack.pop()
            node = Node(node_type="section", text=tb.text, level=level)
            section_stack[-1].add_child(node)
            section_stack.append(node)
            continue

        if tb.block_type == "list_intro":
            clear_list_context()
            node = Node(node_type="list_intro", text=tb.text)
            section_stack[-1].add_child(node)
            current_intro = node
            continue

        if tb.block_type in {"ordered_list_item", "unordered_list_item"}:
            level   = tb.list_level or 0
            ordered = tb.block_type == "ordered_list_item"

            while list_stack and list_stack[-1][0] > level:
                list_stack.pop()

            if not list_stack or list_stack[-1][0] < level or list_stack[-1][1] != ordered:
                list_node = Node(node_type="list", level=level, ordered=ordered)
                if list_stack and list_stack[-1][2].children:
                    last_item = list_stack[-1][2].children[-1]
                    if last_item.node_type == "list_item":
                        last_item.add_child(list_node)
                    else:
                        current_parent().add_child(list_node)
                else:
                    current_parent().add_child(list_node)
                list_stack.append((level, ordered, list_node))

            list_stack[-1][2].add_child(Node(node_type="list_item", text=tb.text, level=level))
            continue

        item = last_list_item()
        if item is not None:
            item.add_child(Node(node_type="paragraph", text=tb.text))
        else:
            clear_list_context()
            section_stack[-1].add_child(Node(node_type="paragraph", text=tb.text))

    return root


def prune_sections(root: Node, excluded_titles: set[str]) -> None:
    normalized = {clean_text(t).lower() for t in excluded_titles if clean_text(t)}
    if not normalized:
        return
    def recurse(node: Node) -> None:
        kept = []
        for child in node.children:
            if child.node_type == "section" and clean_text(child.text).lower() in normalized:
                continue
            recurse(child)
            kept.append(child)
        node.children = kept
    recurse(root)

## 8) Markdown rendering

In [18]:
def render_table(rows: list[list[str]]) -> list[str]:
    if not rows:
        return []
    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]
    esc  = lambda c: c.replace("|", "\\|")
    lines = [
        "| " + " | ".join(esc(c) for c in rows[0]) + " |",
        "| " + " | ".join(["---"] * max_cols) + " |",
    ]
    for row in rows[1:]:
        lines.append("| " + " | ".join(esc(c) for c in row) + " |")
    return lines


def render_list(list_node: Node, lines: list[str], indent: int = 0) -> None:
    idx = 1
    for item in list_node.children:
        prefix = "  " * indent
        marker = f"{idx}." if list_node.ordered else "-"
        lines.append(f"{prefix}{marker} {item.text}")
        if list_node.ordered:
            idx += 1
        for child in item.children:
            if child.node_type == "paragraph":
                lines.append(f"{prefix}  {child.text}")
            elif child.node_type == "list":
                render_list(child, lines, indent + 1)


def render_node(node: Node, lines: list[str]) -> None:
    if node.node_type == "root":
        for child in node.children:
            render_node(child, lines)
        return

    if node.node_type == "section":
        if lines and lines[-1] != "":
            lines.append("")
        lines.append("#" * node.level + " " + node.text)
        lines.append("")
        for child in node.children:
            render_node(child, lines)
        return

    if node.node_type == "paragraph":
        if lines and lines[-1] != "":
            lines.append("")
        lines.append(node.text)
        lines.append("")
        return

    if node.node_type == "list_intro":
        if lines and lines[-1] != "":
            lines.append("")
        text = node.text if node.text.endswith(":") else node.text + ":"
        lines.append(text)
        lines.append("")
        for child in node.children:
            render_node(child, lines)
        return

    if node.node_type == "table":
        if lines and lines[-1] != "":
            lines.append("")
        lines.extend(render_table(node.table_rows))
        lines.append("")
        return

    if node.node_type == "image":
        if lines and lines[-1] != "":
            lines.append("")
        lines.append(f"[[IMAGE: {node.image_filename}]]")
        lines.append("")
        return

    if node.node_type == "list":
        render_list(node, lines)
        if lines and lines[-1] != "":
            lines.append("")


def tree_to_markdown(root: Node) -> str:
    lines: list[str] = []
    render_node(root, lines)
    text = "\n".join(lines).strip() + "\n"
    return re.sub(r"\n{3,}", "\n\n", text)

## 9) End-to-end pipeline

In [19]:
def convert_docx_to_markdown(
    docx_path: str | Path,
    output_path: str | Path | None = None,
    start_heading_text: str = "Login",
    exclude_headings: set[str] | None = None,
    max_heading_level: int = 3,
    image_dir: str | Path | None = None,
) -> dict:
    """
    Full pipeline: DOCX → clean markdown.
    Returns dict with keys: raw_blocks, typed_blocks, tree, markdown, image_dir.
    """
    docx_path = Path(docx_path)
    if image_dir is None:
        base = Path(output_path).parent if output_path else docx_path.parent
        image_dir = base / "images_C"
    image_dir = Path(image_dir)

    raw_blocks   = extract_raw_blocks(docx_path, image_dir)
    raw_blocks   = normalize_raw_blocks(raw_blocks)
    raw_blocks   = drop_preface(raw_blocks, start_heading_text)
    typed_blocks = classify_blocks(docx_path, raw_blocks, max_heading_level)
    tree         = build_tree(typed_blocks)

    if exclude_headings:
        prune_sections(tree, exclude_headings)

    markdown = tree_to_markdown(tree)

    if output_path:
        Path(output_path).write_text(markdown, encoding="utf-8")

    return {
        "raw_blocks":   raw_blocks,
        "typed_blocks": typed_blocks,
        "tree":         tree,
        "markdown":     markdown,
        "image_dir":    image_dir,
    }

## 10) Run

In [20]:
from pathlib import Path

BASE_DIR    = Path(r"C:\PageIndex\PageIndex")
DOCX_PATH   = BASE_DIR / "Functional Requirements.docx"
OUTPUT_PATH = BASE_DIR / "Functional_Requirements_C_1.md"

result = convert_docx_to_markdown(
    docx_path          = DOCX_PATH,
    output_path        = OUTPUT_PATH,
    start_heading_text = "Login",
    exclude_headings   = {},
    max_heading_level  = 3,
)

print(f"Saved markdown : {OUTPUT_PATH}")
print(f"Images saved to: {result['image_dir']}")
print(f"Typed blocks   : {len(result['typed_blocks'])}")
print()
print(result["markdown"][:3000])

<CT_Body '<w:body>' at 0x2462043da40>
Saved markdown : C:\PageIndex\PageIndex\Functional_Requirements_C_1.md
Images saved to: C:\PageIndex\PageIndex\images_C
Typed blocks   : 285

# Login

Users to get access to application by being added to the Okta user group . SSO Login: All Bain users (added to the above group) have SSO login enabled. Users have access to only limited case(s) which will be controlled by the admin users. Initially this is controlled by running SQL queries to add/modify user access with Admin page to be added post MVP. On Login user will land on Home page with a list of cases accessible to the user or can create a new case. Below is a tentative landing page design

[[IMAGE: functional_requirements_image_001.png]]

# Site Navigation

## Navigation Bar

[[IMAGE: functional_requirements_image_002.png]]

Navigation bar is case specific. It will display only the options depending on the assessments selected by the user.

At the bottom of the navigation bar is the “Tool Se